In [1]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import os
import sys
# sys.path.append('/home/matanyaw/DIP_decoder/voxel_embeddings_ROIs')
from voxel_embeddings_ROIs import ROI_coverage, ROI_tsne_funcs
# Getting my modules
sys.path.append('/home/jonathak/VisualEncoder/Analysis/Brain_maps')
from NIPS_utils import get_hemisphere_indices, get_roi_indices, get_roi_indices_per_hemisphere

import surface_figure.flexible_roi_grid 
# Setting up GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Appending Roman's path
sys.path.append('/home/romanb/PycharmProjects/BrainVisualReconst/')

In [2]:
# Loading the model
encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14_reg')
model = torch.load('/home/jonathak/VisualEncoder/Voxels_Prediction/model_ch128.pth').eval().cuda()

# Testing voxel embeddings
voxel_embeddings = model.voxel_embed # Has shape [315997, 256]

# Getting subject 1 indices

subject = 1

lh_start, lh_end = get_hemisphere_indices(subject, 'lh')
rh_start, rh_end = get_hemisphere_indices(subject, 'rh')    
sub_indices = np.arange(lh_start, rh_end)

voxel_embeddings = voxel_embeddings[sub_indices]

ROI_names = ROI_coverage.get_roi_names(subject=subject)

predefined_ROI_indices = {}

# Creating a dictionary of ROI indices (iterating over copy because we remove ROIs that don't exist)
for ROI in ROI_names.copy():
    
    roi_indices = get_roi_indices(subject, ROI)
    
    if roi_indices is None:
        ROI_names.remove(ROI)
    else:
        predefined_ROI_indices[ROI] = roi_indices


Using cache found in /home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/matanyaw/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


ROI 'mTL-bodies' not found for subject 1
ROI 'mTL-faces' not found for subject 1
ROI 'aTL-faces' not found for subject 1
ROI 'mTL-words' not found for subject 1


In [4]:
import image_montage
stroke_imgs_dir = '/home/matanyaw/DIP_decoder/data/matanya_results/results_25_09_15/run_quick_run_2/img_7/roi_EBA'
root_imgs_dir = '/home/matanyaw/DIP_decoder/data/matanya_results/results_25_09_15/run_quick_run_2/img_7'
roi_cov_dir = '/home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages'
output_path = '/home/matanyaw/DIP_decoder/data/debug_montage.png'

image_montage.create_montage(stroke_imgs_dir, root_imgs_dir, roi_cov_dir, output_path)

Creating montaage with ROI coverages form:  /home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages
Montage saved to /home/matanyaw/DIP_decoder/data/debug_montage.png


In [13]:
roi_coverage_configs = []
CENTER_METHODS = [ 'mean', 'meanshift']
DISCRIMINATION_METHODS = ['nearest_voxels', 'nearest_center']
HEMISPHERES = ['lh', 'rh', 'both']


for hemisphere in HEMISPHERES:
    roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, predefined_ROI_indices_dict=predefined_ROI_indices,
                                                            center_method=None, metric=None, discrimination_method='predefined', hemisphere=hemisphere))
    for center in CENTER_METHODS:
        for disc in DISCRIMINATION_METHODS:
            roi_coverage_configs.append(ROI_coverage.InferRoiCoverageConfig(voxel_embeddings=voxel_embeddings, 
                                                                            predefined_ROI_indices_dict=predefined_ROI_indices,
                                                                            center_method=center, 
                                                                            metric='cosine', 
                                                                            discrimination_method=disc,
                                                                            hemisphere=hemisphere))
for cov in roi_coverage_configs:
    print(cov)

Predefined Left
Mean Cos NV Left
Mean Cos NC Left
Meanshift Cos NV Left
Meanshift Cos NC Left
Predefined Right
Mean Cos NV Right
Mean Cos NC Right
Meanshift Cos NV Right
Meanshift Cos NC Right
Predefined
Mean Cos NV
Mean Cos NC
Meanshift Cos NV
Meanshift Cos NC


In [14]:
for cov in roi_coverage_configs:
    print(cov)
    cov.infer_roi_coverage()

Predefined Left
Mean Cos NV Left
Mean Cos NC Left
Meanshift Cos NV Left
Meanshift Cos NC Left
Predefined Right
Mean Cos NV Right
Mean Cos NC Right
Meanshift Cos NV Right
Meanshift Cos NC Right
Predefined
Mean Cos NV
Mean Cos NC
Meanshift Cos NV
Meanshift Cos NC


In [15]:
DATA_DIR = '/home/matanyaw/DIP_decoder/data'

roi_covs_dir = os.path.join(DATA_DIR, f'one_hemi_roi_coverages')
os.makedirs(roi_covs_dir, exist_ok=True)
for coverage in roi_coverage_configs:
    path = os.path.join(roi_covs_dir, coverage.name + '.pkl')
    coverage.save(path)

In [2]:
# python submit_stroke_jobs.py --imgs jonathans --nimgs 2 --roi_cov_dir /home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages --run sided_stroke_night_run
# !Pray this code runs....
import subprocess
run_name = 'deterministic_run_B'
image_indices = [4, 7, 16, 69]
# image_indices = [16]
rois = ["EBA", "FFA-1", "FFA-2", "OPA", "OWFA"]
# rois = ["EBA"]

command = [
    "python",
    "submit_stroke_jobs.py",
    "--imgs", "all",
    "--nimgs", "1",         # Maximum nubmer of images per job
    # "--roi_cov_dir", "/home/matanyaw/DIP_decoder/data/one_roi",
    "--roi_cov_dir", "/home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages",
    "--run", run_name,
    "--job_name", run_name,
    "--images_indices", *map(str, image_indices),
    "--roi_to_process", *map(str, rois),
    # "--steps_to_do", "1", "2",
    # "--dry_run",

]

# Run the command
subprocess.run(command)

Total images: 4  |  imgs/job: 1.00  |  jobs: 4

--> Submitting chunk 1/4: indices[0:1] = [4]
Submitting with:
  sbatch --parsable --job-name=deterministic_run_B_1of4 --output=/home/matanyaw/DIP_decoder/logs/%j_deterministic_run_B_1of4.out --ntasks=1 --cpus-per-task=8 --mem=80G --gres=gpu:1 --time=08:00:00 --partition=irani_run.q --wrap '/home/matanyaw/miniconda3/envs/amit-env/bin/python -u /home/matanyaw/DIP_decoder/stroke_experimet_CLI.py --run deterministic_run_B --roi_cov_dir /home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages --image_type shared --steps_to_do 1 2 4 --images_indices 4 --create_montage --roi_to_process EBA FFA-1 FFA-2 OPA OWFA'
Submitted JobID: 107566
Log: /home/matanyaw/DIP_decoder/logs/107566_deterministic_run_B_1of4.out

--> Submitting chunk 2/4: indices[1:2] = [7]
Submitting with:
  sbatch --parsable --job-name=deterministic_run_B_2of4 --output=/home/matanyaw/DIP_decoder/logs/%j_deterministic_run_B_2of4.out --ntasks=1 --cpus-per-task=8 --mem=80G --gres=gpu:1 -

CompletedProcess(args=['python', 'submit_stroke_jobs.py', '--imgs', 'all', '--nimgs', '1', '--roi_cov_dir', '/home/matanyaw/DIP_decoder/data/one_hemi_roi_coverages', '--run', 'deterministic_run_B', '--job_name', 'deterministic_run_B', '--images_indices', '4', '7', '16', '69', '--roi_to_process', 'EBA', 'FFA-1', 'FFA-2', 'OPA', 'OWFA'], returncode=0)

In [ ]:
roi_coverages = ROI_coverage.load_coverages('/home/matanyaw/DIP_decoder/data/one_roi')
for cov in roi_coverages: 
    for roi in ['EBA', 'FFA-1']:
        print(cov.get_label(),'\t', roi,  f" size: {cov.get_roi_size(roi)}")

Predefined 	 EBA  size: 6237
Predefined 	 FFA-1  size: 882
Predefined Left 	 EBA  size: 2837
Predefined Left 	 FFA-1  size: 552
Predefined Right 	 EBA  size: 3400
Predefined Right 	 FFA-1  size: 330
Mean Cos NC 	 EBA  size: 4185
Mean Cos NC 	 FFA-1  size: 1001
Mean Cos NC Left 	 EBA  size: 1921
Mean Cos NC Left 	 FFA-1  size: 707
Mean Cos NC Right 	 EBA  size: 2264
Mean Cos NC Right 	 FFA-1  size: 294
Mean Cos NV 	 EBA  size: 6237
Mean Cos NV 	 FFA-1  size: 882
Mean Cos NV Left 	 EBA  size: 2913
Mean Cos NV Left 	 FFA-1  size: 420
Mean Cos NV Right 	 EBA  size: 3324
Mean Cos NV Right 	 FFA-1  size: 462
Meanshift Cos NC 	 EBA  size: 4139
Meanshift Cos NC 	 FFA-1  size: 1031
Meanshift Cos NC Left 	 EBA  size: 1911
Meanshift Cos NC Left 	 FFA-1  size: 704
Meanshift Cos NC Right 	 EBA  size: 2228
Meanshift Cos NC Right 	 FFA-1  size: 327
Meanshift Cos NV 	 EBA  size: 6237
Meanshift Cos NV 	 FFA-1  size: 882
Meanshift Cos NV Left 	 EBA  size: 2905
Meanshift Cos NV Left 	 FFA-1  size: 413
Me